In [ ]:
!pip install -q tensorlake outlines

In [ ]:
%env TENSORLAKE_API_KEY=your_tensorlake_api_key
%env OPENAI_API_KEY=your_openai_api_key

In [ ]:
import outlines
from outlines import Template
from pydantic import BaseModel, Field
from tensorlake.documentai import DocumentAI, ParseStatus
import openai

## Step 1: Parse the Document with Tensorlake

In [ ]:
from tensorlake.documentai import DocumentAI, ParseStatus

doc_ai = DocumentAI()
file_id = doc_ai.upload("invoice.pdf")
result = doc_ai.parse_and_wait(file_id)

assert result.status == ParseStatus.SUCCESSFUL

In [ ]:
# Combine the parsed chunks into a structured text
document_text = '\n'.join([
    f"{frag.content}"
    for frag in result.chunks
])

## Step 2: Define the Schema

In [ ]:
from pydantic import BaseModel, Field

class Invoice(BaseModel):
    invoice_number: str = Field(description="Invoice number on the invoice")
    issue_date: str = Field(description="Date when invoice was issued")
    due_date: str = Field(description="Payment due date")
    vendor_name: str = Field(description="Name of the vendor/seller")
    total_amount: float = Field(description="Total amount to be paid")

## Step 3: Create a template for extraction with examples

In [ ]:
# This helps the model understand the extraction pattern
examples = [
    {
        "document": "Invoice #: INV-1234\nDate: 2025-01-15\nDue: 2025-02-15\nVendor: Tech Solutions Inc.\nTotal: $5,250.00",
        "json": '{"invoice_number": "INV-1234", "issue_date": "2025-01-15", "due_date": "2025-02-15", "vendor_name": "Tech Solutions Inc.", "total_amount": 5250.00}'
    },
    {
        "document": "Invoice Number: 2024-0567\nIssued: March 10, 2024\nPayment Due: April 10, 2024\nFrom: Office Supplies Co.\nAmount Due: 1,875.50",
        "json": '{"invoice_number": "2024-0567", "issue_date": "2024-03-10", "due_date": "2024-04-10", "vendor_name": "Office Supplies Co.", "total_amount": 1875.50}'
    },
    {
        "document": "Bill No: B-789\nBilling Date: 05/20/2025\nDue By: 06/20/2025\nCompany: Marketing Agency LLC\nTotal Amount: $12,000",
        "json": '{"invoice_number": "B-789", "issue_date": "2025-05-20", "due_date": "2025-06-20", "vendor_name": "Marketing Agency LLC", "total_amount": 12000.00}'
    }
]

## Step 4: Create the template (following the exact pattern from the example)

In [ ]:
invoice_extraction_prompt = Template.from_string(
    """
    {% for example in examples %}
    DOCUMENT: {{example.document}}
    JSON: {{example.json}}
    {% endfor %}
    DOCUMENT: {{document}}
    JSON:"""
)

## Step 5: Generate the prompt with your document

In [ ]:
prompt = invoice_extraction_prompt(document=document_text, examples=examples)

## Step 6: Use Outlines with OpenAI

In [ ]:
model = outlines.from_openai(openai.OpenAI(), "gpt-4o-mini")
answer = model(prompt)

In [ ]:
answer

'```json\n{\n  "invoice_number": "147NB6",\n  "issue_date": "2018-01-18",\n  "due_date": null,\n  "vendor_name": "Enterprise",\n  "total_amount": 0.00\n}\n```'